In [9]:
import requests
import gzip
import json
import pandas as pd
import os

url = "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Cell_Phones_and_Accessories_5.json.gz"

file_name = "reviews_Cell_Phones_and_Accessories_5.json.gz"

# Download
print("Downloading dataset...")

response = requests.get(
    url,
    stream=True,
    timeout=120
)

print("Status:", response.status_code)

response.raise_for_status()

with open(file_name, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("Download completed!")
print("File size:",
      round(os.path.getsize(file_name) / (1024 * 1024), 2),
      "MB")


# Extract first 20,000 reviews
reviews = []

print("Reading reviews...")

with gzip.open(file_name, "rt", encoding="utf-8") as f:

    for i, line in enumerate(f):

        if i >= 20000:
            break

        review = json.loads(line)

        reviews.append({
            "review_id": i + 1,
            "product_id": review.get("asin"),
            "reviewer_id": review.get("reviewerID"),
            "review_text": review.get("reviewText"),
            "review_title": review.get("summary"),
            "rating": review.get("overall"),
            "helpful_votes": review.get("helpful"),
            "review_date": review.get("reviewTime")
        })


# DataFrame
df = pd.DataFrame(reviews)

# Remove empty reviews
df = df.dropna(subset=["review_text"])

# Create sentiment
def get_sentiment(rating):

    if rating <= 2:
        return "Negative"

    elif rating == 3:
        return "Neutral"

    else:
        return "Positive"


df["sentiment"] = df["rating"].apply(get_sentiment)


# Save
df.to_csv(
    "amazon_mobile_reviews_20000.csv",
    index=False,
    encoding="utf-8-sig"
)


# Results
print("\n==============================")
print("DATASET CREATED")
print("==============================")

print("Total rows:", len(df))

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nFirst 5 rows:")
print(df.head())

Status: 200
Download completed!
File size: 43.31 MB
Reading reviews...

DATASET CREATED
Total rows: 20000

Sentiment distribution:
sentiment
Positive    14352
Negative     3359
Neutral      2289
Name: count, dtype: int64

First 5 rows:
   review_id  product_id     reviewer_id  \
0          1  120401325X  A30TL5EWN6DFXT   
1          2  120401325X   ASY55RVNIL0UD   
2          3  120401325X  A2TMXE2AFO7ONB   
3          4  120401325X   AWJ0WZQYMYFQ4   
4          5  120401325X   ATX7CZYFXI1KW   

                                         review_text  \
0  They look good and stick good! I just don't li...   
1  These stickers work like the review says they ...   
2  These are awesome and make my phone look so st...   
3  Item arrived in great time and was in perfect ...   
4  awesome! stays on, and looks great. can be use...   

                                review_title  rating helpful_votes  \
0                                 Looks Good     4.0        [0, 0]   
1                     